# 11 - Extension aux feux de forêt : détecter et détourer les zones brûlées

Proposition du mentor : appliquer la même démarche à un autre type de catastrophe, les feux, avec
un **détourage** des zones touchées. Même chaîne que pour les inondations :

1. deux images Sentinel-2, **avant** et **après** l'événement ;
2. une carte par pixel de ce qui a changé ;
3. une décision à bord (**ALERTE**, **PAS D'ALERTE**, **INCERTAIN** si trop de nuages) pour la
   zone et pour chaque vignette de 5 km ;
4. le **contour** des zones brûlées, exporté en GeoJSON (lisible par QGIS, Google Earth...) avec
   la surface de chaque zone en hectares.

Pas de réseau entraîné ici : aucun jeu de données de feux annoté n'a pu être préparé dans le temps
du projet. On utilise l'indice physique de référence pour les feux, le **NBR** (bandes B8 et
B12), et sa différence avant / après, le **dNBR**, avec les seuils de sévérité de l'USGS.
Deux bandes et quelques opérations par pixel : le calcul tient sans difficulté à bord.

**Événement** : le feu de l'Évros (Grèce), du 19 août à début septembre 2023, autour
d'Alexandroupoli et de la forêt de Dadia, l'un des plus grands feux mesurés dans l'Union
européenne. Images Sentinel-2 L2A (réflectance de surface) de la copie publique Microsoft
Planetary Computer.

Exécution : Colab, *Exécution > Tout exécuter* (quelques minutes, pas besoin de GPU).

In [ ]:
import os, subprocess, sys

if not os.path.isdir("one-look"):
    subprocess.run(["git", "clone", "-q", "https://github.com/lemembre05/one-look.git"], check=True)
else:
    subprocess.run(["git", "-C", "one-look", "pull", "-q"], check=True)
sys.path.insert(0, "one-look")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "pyproj",
                "pystac-client", "planetary-computer"], check=True)

import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from pyproj import Transformer
import planetary_computer, pystac_client
from src import feux

## 1. Zone et images avant / après

In [ ]:
CENTRE = (41.00, 26.05)          # (latitude, longitude) : entre Alexandroupoli et la forêt de Dadia
COTE_M = 30000                   # zone de 30 km x 30 km
PERIODES = {"avant": "2023-07-20/2023-08-18", "après": "2023-09-10/2023-10-15"}

cat = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                modifier=planetary_computer.sign_inplace)
lat, lon = CENTRE
dlat, dlon = 0.5 * COTE_M / 111000, 0.5 * COTE_M / (111000 * np.cos(np.radians(lat)))
zone = (lon - dlon, lat - dlat, lon + dlon, lat + dlat)

def couvre(item):
    b = item.bbox
    return b[0] <= zone[0] and b[1] <= zone[1] and b[2] >= zone[2] and b[3] >= zone[3]

def chercher(periode, tuile=None):
    items = list(cat.search(collections=["sentinel-2-l2a"], intersects={"type": "Point", "coordinates": [lon, lat]},
                            datetime=periode, query={"eo:cloud_cover": {"lt": 30}}).items())
    items = [i for i in items if couvre(i) and (tuile is None or i.properties.get("s2:mgrs_tile") == tuile)]
    # d'abord les images sans bord de fauchée (pixels sans donnée), puis les moins nuageuses
    return sorted(items, key=lambda i: (round(i.properties.get("s2:nodata_pixel_percentage", 0)), i.properties["eo:cloud_cover"]))

apres = chercher(PERIODES["après"])[0]
avant = chercher(PERIODES["avant"], apres.properties["s2:mgrs_tile"])[0]
for nom, it in [("avant", avant), ("après", apres)]:
    print(f"{nom:6s} {it.id} | {it.datetime:%d/%m/%Y} | nuages {it.properties['eo:cloud_cover']:.1f} % | "
          f"tuile {it.properties['s2:mgrs_tile']} | sans donnée {it.properties.get('s2:nodata_pixel_percentage', 0):.0f} % | "
          f"traitement {it.properties.get('s2:processing_baseline')}")

In [ ]:
def lire(item, bandes=("B02", "B03", "B04", "B08", "B12", "SCL")):
    """Bandes sur une grille commune à 10 m couvrant la zone ; réflectances en 0-1."""
    with rasterio.open(item.assets["B08"].href) as f:
        crs_item = f.crs                      # projection UTM de la tuile
    x, y = Transformer.from_crs("EPSG:4326", crs_item, always_xy=True).transform(lon, lat)
    bornes = (x - COTE_M / 2, y - COTE_M / 2, x + COTE_M / 2, y + COTE_M / 2)
    n = COTE_M // 10
    decalage = 1000 if str(item.properties.get("s2:processing_baseline", "0")) >= "04.00" else 0
    sortie = {}
    for b in bandes:
        with rasterio.open(item.assets[b].href) as f:
            rs = Resampling.nearest if b == "SCL" else Resampling.bilinear
            a = f.read(1, window=from_bounds(*bornes, transform=f.transform), out_shape=(n, n), resampling=rs)
            if b == "B08":
                transform = rasterio.transform.from_origin(bornes[0], bornes[3], 10, 10)
        sortie[b] = a if b == "SCL" else np.clip((a.astype(np.float32) - decalage) / 10000, 0, 1.5)
    return sortie, transform, crs_item.to_string()

images = {}
for nom, it in [("avant", avant), ("après", apres)]:
    images[nom], transform, crs = lire(it)
    print(nom, images[nom]["B08"].shape, f"PIR médian {np.median(images[nom]['B08']):.3f}")

def rgb(im):
    x = np.stack([im["B04"], im["B03"], im["B02"]], -1)
    return np.clip(x / 0.18, 0, 1) ** 0.8

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
for a, nom in zip(ax, images):
    a.imshow(rgb(images[nom])); a.set_title(f"{nom} ({(avant if nom == 'avant' else apres).datetime:%d/%m/%Y})"); a.axis("off")
plt.tight_layout(); plt.show()

## 2. Carte de sévérité (dNBR) et zones brûlées

Pixels exclus : nuages, ombres de nuages, cirrus, pixels saturés ou sans donnée (carte de scène
SCL des produits L2A) sur l'une des deux dates, et l'eau (la mer n'a pas de NBR significatif).
Une zone brûlée = dNBR ≥ 0,27 (sévérité au moins « modérée-faible »), trous bouchés, taches de
moins de 1 ha retirées.

In [ ]:
av, ap = images["avant"], images["après"]
invalides = np.isin(av["SCL"], list(feux.SCL_INVALIDES)) | np.isin(ap["SCL"], list(feux.SCL_INVALIDES))
eau = (av["SCL"] == feux.SCL_EAU) | (ap["SCL"] == feux.SCL_EAU)
valides = ~invalides & ~eau
d = feux.dnbr(av["B08"], av["B12"], ap["B08"], ap["B12"])
brule = feux.carte_brulee(d, valides)
sev = np.where(valides, feux.severite(d), -1)

part_brulee = brule.sum() / max(valides.sum(), 1)
part_invalide = invalides[~eau].mean()
decision = feux.decider_feu(part_brulee, part_invalide)
print(f"Zone de {COTE_M // 1000} km x {COTE_M // 1000} km : {decision}")
print(f"  surface brûlée détectée : {brule.sum() * 100 / 1e4:,.0f} ha ({part_brulee:.1%} des pixels valides)")
print(f"  pixels inutilisables (nuages, ombres...) : {part_invalide:.1%}")
print(pd.Series({nom: int((sev == i).sum() * 100 / 1e4) for i, nom in enumerate(feux.NOMS_SEVERITE)},
                name="surface (ha) par sévérité").to_string())

In [ ]:
from matplotlib.colors import ListedColormap
couleurs = ListedColormap(["#1a9850", "#fee08b", "#fdae61", "#f46d43", "#a50026"])
fig, ax = plt.subplots(1, 3, figsize=(21, 7.5))
ax[0].imshow(rgb(ap)); ax[0].set_title("Après (vraies couleurs)")
im = ax[1].imshow(np.ma.masked_less(sev, 0), cmap=couleurs, vmin=-0.5, vmax=4.5)
ax[1].set_title("Sévérité (dNBR, seuils USGS)")
cb = plt.colorbar(im, ax=ax[1], ticks=range(5), fraction=0.046); cb.ax.set_yticklabels(feux.NOMS_SEVERITE)
ax[2].imshow(rgb(ap)); ax[2].contour(brule, levels=[0.5], colors="red", linewidths=1.2)
ax[2].set_title(f"Détourage des zones brûlées : {decision} ({part_brulee:.0%})")
for a in ax:
    a.axis("off")
plt.tight_layout(); os.makedirs("resultats", exist_ok=True)
plt.savefig("resultats/feux_evros.png", dpi=130); plt.show()

## 3. Décision vignette par vignette (5 km)

Comme pour les inondations : chaque vignette reçoit sa propre décision, seules les vignettes en
ALERTE seraient envoyées au sol en priorité.

In [ ]:
V = 500                                         # 5 km à 10 m
n = brule.shape[0] // V
COULEURS = {"ALERTE": "#c53030", "PAS D'ALERTE": "#2f855a", "INCERTAIN": "#dd6b20"}
lignes = []
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(rgb(ap)); ax.contour(brule, levels=[0.5], colors="red", linewidths=0.8)
for i in range(n):
    for j in range(n):
        sl = (slice(i * V, (i + 1) * V), slice(j * V, (j + 1) * V))
        v, b, inv = valides[sl], brule[sl], invalides[sl] & ~eau[sl]
        if (~eau[sl]).mean() < 0.2:
            continue                             # vignette presque entièrement en mer
        pb = b.sum() / max(v.sum(), 1); pi = inv.sum() / max((~eau[sl]).sum(), 1)
        dec = feux.decider_feu(pb, pi)
        lignes.append({"vignette": f"{i},{j}", "decision": dec, "brule": round(float(pb), 3), "nuages": round(float(pi), 2)})
        ax.add_patch(plt.Rectangle((j * V + 6, i * V + 6), V - 12, V - 12, fill=False, lw=2.5, ec=COULEURS[dec]))
        ax.text(j * V + 15, i * V + 45, dec, color="white", fontsize=8,
                bbox=dict(facecolor=COULEURS[dec], alpha=0.8, pad=2))
ax.axis("off"); plt.tight_layout(); plt.savefig("resultats/feux_vignettes.png", dpi=130); plt.show()
vignettes = pd.DataFrame(lignes)
print(vignettes["decision"].value_counts().to_string())

## 4. Contours exportés (GeoJSON)

Chaque zone brûlée de plus de 1 ha devient un polygone avec sa surface. C'est ce qu'un service de
secours ou de cartographie utiliserait directement.

In [ ]:
geo = feux.contours(brule, transform, crs)
with open("resultats/feux_evros_contours.geojson", "w") as f:
    json.dump(geo, f)
surfaces = pd.Series([e["properties"]["surface_ha"] for e in geo["features"]])
print(f"{len(surfaces)} zones brûlées de plus de 1 ha, {surfaces.sum():,.0f} ha au total")
print(f"plus grande zone : {surfaces.max():,.0f} ha ; 5 plus grandes : {', '.join(f'{s:,.0f}' for s in surfaces.head(5))} ha")
print("Fichier : resultats/feux_evros_contours.geojson (projection", crs + ")")

## 5. Limites

- **Pas d'apprentissage** : les seuils du dNBR sont génériques (USGS) ; un réseau entraîné sur des
  feux annotés (par exemple le jeu de données CaBuAr) serait l'étape suivante, avec la même
  méthode que pour les inondations.
- **Faux positifs possibles** : récoltes et labours entre les deux dates font aussi baisser le NBR
  sur les champs. Ici, les deux dates sont choisies à 4 à 8 semaines d'écart pour limiter cet effet.
- **Pas de vérité terrain chargée ici** : la comparaison avec la carte officielle Copernicus
  (service de cartographie d'urgence) reste à faire.

In [ ]:
import shutil
shutil.make_archive("resultats_feux_11", "zip", "resultats")
try:
    from google.colab import files
    files.download("resultats_feux_11.zip")
except ImportError:
    print("Hors de Colab : resultats_feux_11.zip")